# Metajeux-02 : Casser la composition — où la meilleure réponse composée cesse d'être l'équilibre du jeu composé

**Navigation** : [<< Metajeux-01-Open-Games-Lentilles](Metajeux-01-Open-Games-Lentilles-Python.ipynb) | [Version Python](Metajeux-02-Casser-la-Composition-Python.ipynb) | [Version C#](Metajeux-02-Casser-la-Composition-CSharp.ipynb) | [Index](README.md) | [Metajeux-03-Abstraction-a-Dette >>](Metajeux-03-Abstraction-a-Dette-Python.ipynb)

GT-18 a posé la composition d'open games et vérifié la propriété de Hedges — « la meilleure réponse de la composition est la composition des meilleures réponses » — sur un jouet où elle tient, puis mesuré la convergence de la boucle vers son point fixe. Ce notebook fait ce que le critère de maturation demande : **il essaie de casser le jouet, et regarde ce qui casse**.

Deux surfaces d'attaque :

- **Surface A — la sélection n'est pas compositionnelle.** Quand une meilleure réponse locale est *multivoque* (plateau d'optima) et que les paiements sont *couplés*, composer des sélections d'argmax locaux produit un profil qui n'est pas un équilibre du jeu composé. La propriété de Hedges, dans sa forme exacte, porte sur les *correspondances* d'ensembles ; la lecture fonctionnelle du jouet GT-18 — composer des points — ne survit qu'univoque.
- **Surface B — la frontière de contraction.** La convergence de la boucle GT-18 tient parce que l'update est (presque) contractante. En balayant le pas d'update, la boucle passe de la convergence monotone au 2-cycle exact, puis à l'oscillation persistante — et la frontière mesurée coïncide avec la borne de Banach, exactement.

Références : Ghani, N., Hedges, J., Winschel, V., Zahn, P. (2018), *Compositional Game Theory* (LICS '18, arXiv:1603.04641) ; Metajeux-01 (`Metajeux-01-Open-Games-Lentilles-Python.ipynb`).

## Objectifs d'apprentissage

1. **Reproduire le régime où la composition tient** : couplée mais univoque, la meilleure réponse composée est bien l'équilibre du composé.
2. **Exhiber la casse** : ajouter UN ingrédient — la multivocité du plateau — et montrer par déviation explicite qu'un profil issu de la composition n'est pas Nash du composé.
3. **Isoler les ingrédients** : le couplage seul ne casse rien (cellule 3), la multivocité seule non plus (r = 0) ; c'est leur *conjonction* qui brise la sélection.
4. **Mesurer la frontière de contraction** : carte convergence / 2-cycle / oscillation sur le pas d'update, comparée à la borne théorique de Banach `L = |1 - delt| < 1`.
5. **Reconsidérer honnêtement le jouet GT-18** : sa boucle additive n'est jamais contractante (L = 1 partout) — sa convergence est une convergence-au-rail, pas un point fixe de Banach.

## 1. Le régime où la composition tient — couplée mais univoque

Rappel du cadre GT-18 : deux open games composés en série. Le joueur 1 choisit `x`, le joueur 2 observe `x` et choisit `y`. Le paiement de J2 dépend de `x` (couplage), celui de J1 de `y` (couplage en retour). Ici chaque meilleure réponse est **univoque** : `u1 = -(x-1)^2` a un unique maximum en `x = 1`, et `u2 = -(y-x)^2` a un unique maximum en `y = x`. La composition des argmax donne le profil `(1, 1)`, et ce profil est bien l'équilibre du jeu composé — vérifié par énumération des déviations, pas par affirmation.

In [2]:
// === Deux open games en serie, COUPLES mais UNIVOQUES ===
const ACTIONS_X: [i64; 3] = [0, 1, 2];
const ACTIONS_Y: [i64; 3] = [0, 1, 2];

fn sq(x: f64) -> f64 { x * x }
fn u1(x: i64, y: i64) -> f64 { -sq((x - 1) as f64) + 0.0 * y as f64 }
fn u2(x: i64, y: i64) -> f64 { -sq((y - x) as f64) }

fn argmax_first(actions: &[i64], f: impl Fn(i64) -> f64) -> i64 {
    let mut best = actions[0];
    let mut best_val = f(best);
    for &a in &actions[1..] {
        let v = f(a);
        if v > best_val { best = a; best_val = v; }
    }
    best
}
fn br1(y: i64) -> i64 { argmax_first(&ACTIONS_X, |x| u1(x, y)) }
fn br2(x: i64) -> i64 { argmax_first(&ACTIONS_Y, |y| u2(x, y)) }

// Equilibre du compose : enumeration exhaustive des deviations
fn est_nash_compose(x: i64, y: i64) -> bool {
    let ok1 = ACTIONS_X.iter().all(|&xp| u1(x, y) >= u1(xp, br2(xp)) - 1e-12);
    let ok2 = ACTIONS_Y.iter().all(|&yp| u2(x, y) >= u2(x, yp) - 1e-12);
    ok1 && ok2
}

fn py(b: bool) -> &'static str { if b { "True" } else { "False" } }

// Composition des selections : x* = br1(y exogene), puis y* = br2(x*)
let x_star = br1(0);
let y_star = br2(x_star);
let profil_compose = (x_star, y_star);

println!("BR locales univoques : br1(0) = {}, br2({}) = {}", br1(0), x_star, y_star);
println!("Profil issu de la composition : ({}, {})", profil_compose.0, profil_compose.1);
println!("Ce profil est-il Nash du compose : {}", py(est_nash_compose(profil_compose.0, profil_compose.1)));
println!("Regime univoque-couple : la composition des selections Donne l'equilibre du compose.");

BR locales univoques : br1(0) = 1, br2(1) = 1


Profil issu de la composition : (1, 1)


Ce profil est-il Nash du compose : True


Regime univoque-couple : la composition des selections Donne l'equilibre du compose.


**Lecture de la sortie committée** : la composition des argmax produit `(1, 1)`, et l'énumération exhaustive des déviations confirme que c'est l'équilibre du composé. Remarquer ce qui est déjà là : les paiements sont couplés dans les deux sens (`u2` dépend de `x`), et pourtant la composition tient. **Le couplage seul ne casse pas la sélection** — c'est un contre-fait utile : il faut autre chose. C'est cet « autre chose » que la section 2 installe.

## 2. Surface A — la multivocité du plateau brise la sélection

On déplace le maximum de J1 au **centre exact d'un plateau** : `u1 = -(x - 0.5)^2 + r * y` avec `r = 0.3`. Sur `x ∈ {0, 1, 2}`, les valeurs locales sont `-0.25, -0.25, -2.25` — `x = 0` et `x = 1` sont **ex æquo** : la meilleure réponse locale est *multivoque* (`BR = {0, 1}`, plateau par symétrie exacte). La multivocité seule serait inoffensive : avec `r = 0`, J1 resterait indifférent entre les deux profils composés. Mais le couplage `r * y` fait que le choix de J1, via la réponse `y = x` de J2, **revient à J1** : deviner `x = 1` rapporte `r` de plus que `x = 0`. La composition naïve — prendre UN élément du plateau, disons le plus petit — rend un profil que le jeu composé réfute.

In [3]:
// === Surface A : plateau multivoque + couplage ===
const ACTIONS_X: [i64; 3] = [0, 1, 2];
const ACTIONS_Y: [i64; 3] = [0, 1, 2];
const C_PLATEAU: f64 = 0.5;
const R_COUPLAGE: f64 = 0.3;

fn sq(x: f64) -> f64 { x * x }
fn u2(x: i64, y: i64) -> f64 { -sq((y - x) as f64) }
fn argmax_first(actions: &[i64], f: impl Fn(i64) -> f64) -> i64 {
    let mut best = actions[0];
    let mut best_val = f(best);
    for &a in &actions[1..] {
        let v = f(a);
        if v > best_val { best = a; best_val = v; }
    }
    best
}
fn br2(x: i64) -> i64 { argmax_first(&ACTIONS_Y, |y| u2(x, y)) }

fn u1_a(x: i64, y: f64) -> f64 { -sq(x as f64 - C_PLATEAU) + R_COUPLAGE * y }

fn br1_a_ensemble() -> (Vec<i64>, Vec<f64>) {
    let vals: Vec<f64> = ACTIONS_X.iter().map(|&x| u1_a(x, 0.0)).collect();
    let vmax = vals.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
    let br: Vec<i64> = ACTIONS_X.iter().cloned().filter(|&x| (u1_a(x, 0.0) - vmax).abs() < 1e-12).collect();
    (br, vals)
}

fn str_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn int_list(items: &[i64]) -> String {
    format!("[{}]", items.iter().map(|&x| x.to_string()).collect::<Vec<_>>().join(", "))
}
fn pair_list(items: &[(i64, i64)]) -> String {
    format!("[{}]", items.iter().map(|&(a, b)| format!("({}, {})", a, b)).collect::<Vec<_>>().join(", "))
}

let (br1_a, vals_locales) = br1_a_ensemble();
let txt: Vec<String> = vals_locales.iter().map(|v| format!("{:+.3}", v)).collect();
println!("Valeurs locales de u1 (y exogene) : {}", str_list(&txt));
println!("BR locale de J1 = {}  (MULTIVOQUE : plateau par symetrie exacte c=0.5)", int_list(&br1_a));

// Composition des selections : on prend UN element du plateau (le plus petit)
let selection = br1_a.iter().copied().min().expect("plateau non vide");
let profils_composes: Vec<(i64, i64)> = br1_a.iter().map(|&x| (x, br2(x))).collect();
println!("Profil de la composition naive (plus petit argmax) : ({}, {})", selection, br2(selection));
println!("Ensemble prevu par la composition des correspondances : {}", pair_list(&profils_composes));

Valeurs locales de u1 (y exogene) : ['-0.250', '-0.250', '-2.250']


BR locale de J1 = [0, 1]  (MULTIVOQUE : plateau par symetrie exacte c=0.5)


Profil de la composition naive (plus petit argmax) : (0, 0)


Ensemble prevu par la composition des correspondances : [(0, 0), (1, 1)]


**Lecture de la sortie committée** : la correspondance locale de J1 est `{0, 1}` — le plateau est exact, produit de la symétrie `c = 0.5` au milieu de deux actions consécutives. La composition des correspondances prévoit l'ensemble `{(0,0), (1,1)}`. Reste à demander au jeu composé ce qu'il en pense : chaque profil prédit est confronté à toutes les déviations, J1 anticipant la réponse de J2.

In [4]:
// === Test Nash du compose, profil par profil ===
const ACTIONS_X: [i64; 3] = [0, 1, 2];
const ACTIONS_Y: [i64; 3] = [0, 1, 2];
const C_PLATEAU: f64 = 0.5;
const R_COUPLAGE: f64 = 0.3;

fn sq(x: f64) -> f64 { x * x }
fn u2(x: i64, y: i64) -> f64 { -sq((y - x) as f64) }
fn argmax_first(actions: &[i64], f: impl Fn(i64) -> f64) -> i64 {
    let mut best = actions[0];
    let mut best_val = f(best);
    for &a in &actions[1..] {
        let v = f(a);
        if v > best_val { best = a; best_val = v; }
    }
    best
}
fn br2(x: i64) -> i64 { argmax_first(&ACTIONS_Y, |y| u2(x, y)) }
fn u1_a(x: i64, y: f64) -> f64 { -sq(x as f64 - C_PLATEAU) + R_COUPLAGE * y }
fn br1_a_ensemble() -> (Vec<i64>, Vec<f64>) {
    let vals: Vec<f64> = ACTIONS_X.iter().map(|&x| u1_a(x, 0.0)).collect();
    let vmax = vals.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
    let br: Vec<i64> = ACTIONS_X.iter().cloned().filter(|&x| (u1_a(x, 0.0) - vmax).abs() < 1e-12).collect();
    (br, vals)
}

fn ecart_deviation_j1(x: i64, y: i64) -> Option<f64> {
    if y != br2(x) { return None; } // le profil ne respecte meme pas la BR de J2
    let best = ACTIONS_X.iter().map(|&xp| u1_a(xp, br2(xp) as f64)).fold(f64::NEG_INFINITY, f64::max);
    Some(best - u1_a(x, y as f64))
}

let (br1_a, _) = br1_a_ensemble();
let profils_composes: Vec<(i64, i64)> = br1_a.iter().map(|&x| (x, br2(x))).collect();

println!("{:<14} {:>9} {:>23} {:>8}  Verdict", "Profil prevu", "u1(x, y)", "meilleure deviation J1", "ecart");
println!("{}", "-".repeat(78));
for &(x, y) in &profils_composes {
    let ecart = ecart_deviation_j1(x, y);
    let x_best = argmax_first(&ACTIONS_X, |xp| u1_a(xp, br2(xp) as f64));
    let verdict = if matches!(ecart, Some(e) if e > 1e-9) { "NON-NASH du compose (J1 devie)" } else { "equilibre du compose" };
    let e = ecart.expect("ecart non defini : le profil ne respecte pas br2");
    println!("({}, {}){:<7} {:9.3} ({}, {}){:<12} {:+8.3}  {}", x, y, "", u1_a(x, y as f64), x_best, br2(x_best), "", e, verdict);
}

// Contre-fait : R_COUPLAGE = 0 -> le plateau ne casse plus rien
fn u1_r0(x: i64, _y: i64) -> f64 { -sq(x as f64 - C_PLATEAU) }
let ecarts_r0: Vec<f64> = profils_composes.iter()
    .map(|&(x, _y)| ACTIONS_X.iter().map(|&xp| u1_r0(xp, br2(xp))).fold(f64::NEG_INFINITY, f64::max) - u1_r0(x, br2(x)))
    .collect();
println!();
println!("Contre-fait r=0 (plateau sans couplage) : ecarts = [{:+.3}, {:+.3}] -> les deux profils survivent (indifference, equilibres faibles)",
         ecarts_r0[0], ecarts_r0[1]);

Profil prevu    u1(x, y)  meilleure deviation J1    ecart  Verdict


------------------------------------------------------------------------------


(0, 0)           -0.250 (1, 1)               +0.300  NON-NASH du compose (J1 devie)


(1, 1)            0.050 (1, 1)               +0.000  equilibre du compose


Contre-fait r=0 (plateau sans couplage) : ecarts = [+0.000, +0.000] -> les deux profils survivent (indifference, equilibres faibles)


**Lecture de la sortie committée** : la casse est exhibée par déviation explicite. Le profil `(0, 0)` — prédit par la composition naïve (sélection du plus petit argmax du plateau) — **n'est pas Nash du composé** : J1, en anticipant que J2 répondra `y = x`, gagne `+0.300` à dévier vers `x = 1`. Seul `(1, 1)` survit. Et le contre-fait `r = 0` confirme l'isolation des ingrédients : sans couplage, le plateau est inoffensif (les deux profils sont des équilibres faibles par indifférence).

**Ce que Hedges garantise vraiment.** La propriété de composition, dans sa forme exacte, porte sur les **correspondances** — les ensembles de meilleures réponses — munies de leur contexte : la composée des correspondances calcule bien l'ensemble des solutions du jeu composé, *à contexte correctement propagé*. Ce qui ne tient pas, et que le jouet univoque de GT-18 ne pouvait pas montrer, est la lecture **fonctionnelle** : composer des *sélections* d'argmax locaux, chacune calculée sans regarder le couplage en retour, ne prédit pas l'équilibre — la composition prédit l'ensemble (sur-approximation), pas le point. Chaque ingrédient est inoffensif seul — couplage sans plateau (section 1), plateau sans couplage (contre-fait r = 0) — c'est leur conjonction qui brise la sélection.

## 3. Surface B — la frontière de contraction de la boucle

La section 3 de GT-18 itère une boucle `W -> P -> decision -> update -> W'` et mesure la distance au point fixe. La famille d'update la plus simple qui généralise la sienne est le **rappel homing** : `a' = clip(a + delt * (b - a), -1, 1)`, qui tire `a` vers la cible `b` avec un pas `delt`. Cette application est affine de pente `1 - delt` (donc de constante de Lipschitz `L = |1 - delt|`) tant que le clip ne mord pas : le **théorème du point fixe de Banach** garantit contraction, donc convergence géométrique vers `b`, si et seulement si `L < 1`, c'est-à-dire `delt ∈ (0, 2)`. À `delt = 2` exactement, `L = 1` : la boucle rebondit de part et d'autre de la cible sans amortir — un 2-cycle exact. Au-delà, chaque pas **amplifie** l'écart (`a - b -> (1-delt)(a - b)` avec `|1-delt| > 1`) jusqu'à atteindre le rail `|a| = 1`, où le clip borne la divergence en un **cycle de période 2 collé au rail** — bornée, la boucle ne converge pas pour autant.

La prédiction théorique est nette : la frontière entre convergence et non-convergence est **exactement `delt = 2`**. On la mesure.

In [5]:
// === Surface B : carte delt -> comportement de la boucle ===
const B_CIBLE: f64 = 0.3;

fn clip(x: f64, lo: f64, hi: f64) -> f64 { x.max(lo).min(hi) }

fn boucle_homing(a0: f64, delt: f64, b: f64, n: usize) -> Vec<f64> {
    let mut traj = vec![a0];
    let mut a = a0;
    for _ in 0..n {
        a = clip(a + delt * (b - a), -1.0, 1.0);
        traj.push(a);
    }
    traj
}

fn allclose(a: &[f64], b: f64, atol: f64) -> bool {
    const RTOL: f64 = 1e-05; // defaut de numpy.allclose
    a.iter().all(|&x| (x - b).abs() <= atol + RTOL * b.abs())
}

fn classifie(traj: &[f64], b: f64, tol: f64) -> &'static str {
    let queue = &traj[traj.len() - 20..];
    if queue.iter().all(|&q| (q - b).abs() < 1e-4) { return "converge"; }
    let ecarts: Vec<f64> = queue.iter().map(|&q| q - b).collect();
    let pairs: Vec<f64> = ecarts.iter().step_by(2).cloned().collect();
    let impairs: Vec<f64> = ecarts.iter().skip(1).step_by(2).cloned().collect();
    if allclose(&pairs, ecarts[0], tol) && allclose(&impairs, ecarts[1], tol) {
        let sur_rail = queue.iter().any(|&q| (q.abs() - 1.0).abs() < 1e-9);
        return if sur_rail { "2-cycle sur rail" } else { "2-cycle interieur" };
    }
    "non stabilise"
}

const DELTAS: [f64; 8] = [0.5, 1.0, 1.5, 1.9, 2.0, 2.1, 2.5, 3.0];
println!("{:>5} {:>10} {:>9} {:>10}  comportement", "delt", "L=|1-delt|", "a_120", "|a_120-b|");
println!("{}", "-".repeat(58));
for delt in DELTAS {
    let traj = boucle_homing(0.9, delt, B_CIBLE, 120);
    let comp = classifie(&traj, B_CIBLE, 1e-6);
    let last = traj[traj.len() - 1];
    println!("{:5.1} {:10.2} {:9.4} {:10.4}  {}", delt, (1.0 - delt).abs(), last, (last - B_CIBLE).abs(), comp);
}
println!();
println!("Borne de Banach : contraction ssi L < 1 ssi delt dans (0, 2)");
println!("Frontiere mesuree : convergence pour delt <= 1.9, bascule a delt = 2.0 exactement.");

 delt L=|1-delt|     a_120  |a_120-b|  comportement


----------------------------------------------------------


  0.5       0.50    0.3000     0.0000  converge


  1.0       0.00    0.3000     0.0000  converge


  1.5       0.50    0.3000     0.0000  converge


  1.9       0.90    0.3000     0.0000  converge


... (2 outputs left out)

  2.5       1.50    1.0000     0.7000  2-cycle sur rail


  3.0       2.00    1.0000     0.7000  2-cycle sur rail


Borne de Banach : contraction ssi L < 1 ssi delt dans (0, 2)


Frontiere mesuree : convergence pour delt <= 1.9, bascule a delt = 2.0 exactement.


**Lecture de la sortie committée** : la frontière mesurée coïncide avec la borne théorique **exactement**. En dessous de `delt = 2` (`L < 1`), la boucle converge vers la cible `b = 0.3` — géométriquement vite près de `delt = 1` (`L = 0`, convergence en un pas), plus lentement aux bords. À `delt = 2.0` (`L = 1`, cas limite non contractant), la boucle rebondit : l'écart change de signe à chaque pas sans changer de magnitude — le **2-cycle exact** prédit par l'analyse. Au-delà (`L > 1`), l'écart s'amplifie à chaque pas jusqu'à mordre le rail `|a| = 1` ; le clip a alors remplacé la divergence par un **2-cycle sur rail** — une des deux valeurs du cycle est collée au mur — la boucle bornée n'est plus pour autant convergente. Le clip ne répare pas la perte de contraction : il en masque la conséquence visible.

## 4. Le régime exact de la boucle GT-18 — jamais contractante

Reconsidérons la boucle effectivement committée dans GT-18 : `a' = clip(a + delta, -1, 1)` — un pas **additif** constant. Sa pente est `1` partout où le clip ne mord pas : constante de Lipschitz `L = 1`, **le cas limite non contractant, pour toute valeur de `delta`**. Banach ne s'applique pas ; ce qui fait converger la boucle de GT-18 vers `1.0` (cas `delta = 0.5`) est uniquement le **rail** : chaque pas ajoute `delta`, le clip tronque à `1`, et la valeur s'y colle — c'est une convergence-au-rail en temps borné (au plus `2/delta` pas), pas une convergence géométrique vers un point fixe intérieur. La distinction n'est pas cosmétique : la convergence-au-rail exige un mur pour s'arrêter, la convergence de Banach non — la surface B vient de le mesurer (`delt > 2` : même avec un rail, pas d'arrêt).

In [6]:
// === GT-18 reframed : la boucle additive est le cas L=1 partout ===
const B_CIBLE: f64 = 0.3;

fn clip(x: f64, lo: f64, hi: f64) -> f64 { x.max(lo).min(hi) }

fn boucle_homing(a0: f64, delt: f64, b: f64, n: usize) -> Vec<f64> {
    let mut traj = vec![a0];
    let mut a = a0;
    for _ in 0..n {
        a = clip(a + delt * (b - a), -1.0, 1.0);
        traj.push(a);
    }
    traj
}

fn boucle_additive(a0: f64, delta: f64, n: usize) -> Vec<f64> {
    let mut traj = vec![a0];
    let mut a = a0;
    for _ in 0..n {
        a = clip(a + delta, -1.0, 1.0);
        traj.push(a);
    }
    traj
}

fn str_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

fn opt_usize(o: Option<usize>) -> String {
    match o { Some(n) => n.to_string(), None => "None".to_string() }
}

println!("Boucle additive GT-18 (L=1 partout, jamais contractante) :");
for delta in [0.5, -0.3] {
    let traj = boucle_additive(0.5, delta, 120);
    let rail = if delta > 0.0 { 1.0 } else { -1.0 };
    let n_rail = traj.iter().position(|&a| (a - rail).abs() < 1e-12);
    println!("  delta={:+.1} : atteint le rail en {} pas, y reste (a_120 = {:+.1})", delta, opt_usize(n_rail), traj[traj.len() - 1]);
}
println!();
println!("Boucle homing contractante (delt=0.8, L=0.2) : convergence GEOMETRIQUE interieure");
let traj = boucle_homing(0.9, 0.8, B_CIBLE, 120);
let errs: Vec<f64> = traj[..8].iter().map(|&a| (a - B_CIBLE).abs()).collect();
let txt: Vec<String> = errs.iter().map(|e| format!("{:.4}", e)).collect();
println!("  |a_n - b| pour n=0..7 : {}", str_list(&txt));
println!("  ratio moyen d'amortissement par pas : {:.3} (theorie L = 0.2)", (errs[7] / errs[0]).powf(1.0 / 7.0));
println!();
println!("Convergence-au-rail : temps borne par le mur, arret PAR le mur.");
println!("Convergence de Banach : taux geometrique |1-delt|, arret par amortissement interne.");

Boucle additive GT-18 (L=1 partout, jamais contractante) :


  delta=+0.5 : atteint le rail en 1 pas, y reste (a_120 = +1.0)


  delta=-0.3 : atteint le rail en 5 pas, y reste (a_120 = -1.0)


Boucle homing contractante (delt=0.8, L=0.2) : convergence GEOMETRIQUE interieure


  |a_n - b| pour n=0..7 : ['0.6000', '0.1200', '0.0240', '0.0048', '0.0010', '0.0002', '0.0000', '0.0000']


  ratio moyen d'amortissement par pas : 0.200 (theorie L = 0.2)


Convergence-au-rail : temps borne par le mur, arret PAR le mur.


Convergence de Banach : taux geometrique |1-delt|, arret par amortissement interne.


**Lecture de la sortie committée** : les deux dynamiques sont distinguées par leur signature. La boucle additive de GT-18 atteint le rail en un nombre entier de pas (`2/delta` ici : 1 pas pour `delta = 0.5` partant de 0.5) et s'y colle — l'arrêt est un événement *discret*, causé par le mur. La boucle homing contractante, elle, amortit son erreur d'un facteur `L` à chaque pas — mesuré `≈ 0.2`, théorie `0.2` — et l'arrêt est *interieur*, asymptotique, sans mur. Le jouet GT-18 vivait donc dans le cas dégénéré `L = 1` : sa « convergence vers le point fixe » était la sédimentation sur le rail, et la section 3 de GT-18 mesurait la distance à un « point fixe » qui n'en était un que par clip.

## 5. Verdict par surface — ce qui cède, ce qui tient

| Surface | Ce qui cède | Ce qui tient | Condition exacte |
|---|---|---|---|
| **A — sélection compositionnelle** | composer des *sélections* d'argmax locaux sous plateau multivoque + couplage : le profil `(0,0)` prédit n'est pas Nash (`écart +0.300`) | la propriété de Hedges au sens des **correspondances** (l'ensemble composé sur-approxime les équilibres : `(1,1)` y est) ; la composition des sélections en régime **univoque** (section 1) | la casse exige la conjonction plateau × couplage — chacun seul est inoffensif (section 1, contre-fait r = 0) |
| **B — convergence de la boucle** | la lecture « la boucle converge donc elle est saine » : à `L ≥ 1` elle ne converge pas (2-cycle intérieur à `delt = 2`, 2-cycle sur rail au-delà) ; la boucle GT-18 est `L = 1` **partout** — jamais contractante | la frontière de contraction est **exactement** la borne de Banach `delt = 2` — le théorique et le mesuré coïncident sans zone grise | convergence géométrique ssi `delt ∈ (0, 2)` ; convergence-au-rail de GT-18 = arrêt par le mur, pas par amortissement |

Le bilan de maturation est positif au sens strict : le jouet GT-18 **survit borné** — aucun de ses résultats numériques n'était faux — mais son périmètre de validité est désormais écrit : la composition des sélections exige l'univocité (ou l'absence de couplage), et la convergence de sa boucle est de type rail, pas de type Banach. C'est ce que « survivre à une variante -b » veut dire : pas rester intact, avoir ses bornes.

### Questions ouvertes — ce que cette variante ne tranche pas

- **La sélection d'équilibre en amont.** La surface A montre que la composition sur-approxime l'ensemble des équilibres ; elle ne dit pas lequel un processus décentralisé sélectionne — ni si un raffinement (élimination des équilibres faibles par indifférence) suffirait à rétablir une composition des sélections *strictes*.
- **Le plateau exact est une symétrie.** La multivocité exhibée naît de `c = 0.5`, milieu exact de deux actions — un plateau de mesure nulle dans l'espace des paramètres. Un plateau épais (plateau d'utilité plate sur un intervalle, fréquent en jeux discrets de coordination) casserait de la même façon, mais la frontière « plateau de mesure nulle vs épais » n'est pas explorée ici.
- **Le clip lisse-t-il jamais ?** Sur la surface B, le clip a borné la divergence en oscillation. Il existe des update non lisses où le clip produit au contraire des points fixes attractifs *sur* le rail (convergence vers le mur avec approche unilatérale) — la taxonomy complète des régimes `L > 1` bornés n'est pas dressée.

## Exercice 1 : le seuil de couplage

La casse de la surface A exige `r > 0` — mais à quel seuil exactement ? Le profil `(0, 0)` prédit par la composition naïve cesse d'être un équilibre dès que `r` est strictement positif. Vérifiez-le : balayez `r` sur une suite de valeurs incluant zéro et, pour chaque valeur, recalculez l'écart de déviation de J1 au profil `(0, 0)`. Confirmez que l'écart vaut exactement `r` (pourquoi ?) et dites ce que devient le verdict à `r = 0`.

In [7]:
// Exercice 1 a completer
// 1. Pour r dans (0..7).map(|i| 0.6 * i as f64 / 6.0) : reconstruire u1_a avec ce couplage
// 2. Calculer l'ecart de deviation de J1 au profil (0, 0) : max_x u1(x, br2(x)) - u1(0, 0)
// 3. Verifier que l'ecart == r exactement, et expliquer pourquoi (indice : u1(1,1) - u1(0,0) = ... )
// Indice : -(1-0.5)^2 + r*1 - (-(0-0.5)^2 + r*0) = 0 + r = r
println!("Exercice a completer");
let _resultat_ex1: Option<Vec<f64>> = None; // TODO etudiant : ecarts_par_r = ...

Exercice a completer


## Exercice 2 : le 2-cycle en main

À `delt = 2.0` exactement, la théorie prédit que l'écart `a - b` change de signe à chaque pas sans changer de magnitude : `a_{n+1} - b = -(a_n - b)`. Partant de `a0 = 0.9` avec `b = 0.3` : calculez à la main les quatre premiers termes de la suite des écarts, puis vérifiez avec `boucle_homing` que la trajectoire exacte les reproduit. Combien vaut l'amplitude du cycle, et pourquoi ne décroît-elle jamais ?

In [8]:
// Exercice 2 a completer
// 1. Main : a0 = 0.9, b = 0.3, delt = 2.0 -> ecarts a_n - b pour n = 0..3
// 2. Verifier avec boucle_homing(0.9, 2.0) : les 4 premiers ecarts
// 3. Amplitude du cycle = |a0 - b| * (1 - 0) = |a0 - b| : pourquoi constante ?
// Indice : a' - b = clip(a + 2(b-a)) - b = (a + 2b - 2a) - b = -(a - b) tant que le clip ne mord pas
println!("Exercice a completer");
let _resultat_ex2: Option<Vec<f64>> = None; // TODO etudiant : ecarts_theoriques = ...

Exercice a completer


## Exercice 3 : la cible hors du domaine

Toute la surface B vise `b = 0.3`, intérieur au domaine `[-1, 1]` du clip. Que se passe-t-il quand la cible est **hors** du domaine — `b = 1.5` ? Prédisez d'abord (par écrit) : la boucle `a' = clip(a + delt*(b-a))` avec `delt = 0.8` converge-t-elle, et vers quoi ? Vérifiez numériquement : la limite est-elle le point fixe de Banach, ou autre chose ? Mesurez la distance finale et interprétez : le clip **projette** le point fixe sur le domaine fermé.

In [9]:
// Exercice 3 a completer
// 1. Prediction ecrite AVANT simulation : convergence vers b=1.5 ? vers 1.0 ? autre ?
// 2. Simuler boucle_homing(0.9, 0.8, b=1.5, n=120) et mesurer la limite
// 3. Distance finale : |limite - 1.5| = 0.5 -> le point fixe atteignable est projete sur le bord
// Indice : si a < 1 alors a' = a + 0.8*(1.5-a) > a (stritement) et clip(a') = 1 des que a >= 1
println!("Exercice a completer");
let _resultat_ex3: Option<f64> = None; // TODO etudiant : limite_b15 = ...

Exercice a completer


## Conclusion et perspectives

GT-18 a survécu à sa variante -b au sens de la maturation : ses résultats tiennent, **et leurs bornes sont écrites**. Trois acquis :

1. **La sélection n'est pas compositionnelle** — la composition de sélections d'argmax locaux rend un profil non-Nash (`(0, 0)`, écart `+0.300`) dès qu'un plateau multivoque rencontre un couplage ; la propriété de Hedges n'a jamais promis autre chose que la composition des *correspondances*, qui sur-approxime l'ensemble des équilibres. Prédire le point exige l'univocité — ou une théorie de la sélection.
2. **La frontière de contraction est la borne de Banach, exactement** — mesurée à `delt = 2` sans zone grise ; au-delà, le clip borne la divergence en oscillation sur rail, il ne la répare pas.
3. **La boucle GT-18 est le cas dégénéré `L = 1`** — sa convergence est une convergence-au-rail (arrêt par le mur, temps borné), pas une convergence géométrique de Banach (arrêt par amortissement interne). Les deux signatures sont mesurées et distinguées.

Perspectives : le même patron s'applique à chaque objet mathématique porté par un jouet — vérifier la propriété sur le régime où elle tient (GT-18 l'a fait), puis chercher la frontière où elle cesse (ce notebook), puis écrire les bornes dans la prose (fait au §5). Le voisinage immédiat : la sélection d'équilibre dans les jeux composés à contextes multiples, et la taxonomy complète des boucles bornées non contractantes.

## References

- **Ghani, N., Hedges, J., Winschel, V., Zahn, P. (2018).** "Compositional Game Theory." *Proceedings of the 33rd Annual ACM/IEEE Symposium on Logic in Computer Science (LICS '18)*, 472-481, DOI 10.1145/3209108.3209165. — formalise la composition séquentielle et parallèle de jeux dans un cadre catégorique ; fondation de la lecture compositionnelle des correspondances d'équilibre.
- **Sandholm, W. H. (2010).** *Population Games and Evolutionary Dynamics.* MIT Press. — dynamique de population et evolution des équilibres, base théorique des modèles d'équilibre évolutionnaire.
- **Kakutani, S. (1941).** "A generalization of Brouwer's fixed-point theorem." *Duke Mathematical Journal* 8(3), 457-459. — théorème fondateur des correspondances multivoques, base de Nash 1950 et de la multivocité exhibée en Surface A.
- **Govindan, S., Wilson, R. (2003).** "A global Newton method to compute Nash equilibria." *Journal of Economic Theory* 110(1), 65-86, DOI 10.1016/S0022-0531(03)00005-X. — méthode numérique de calcul d'équilibres par homotopie.
- **Capucci, M., Gavranović, B., Hedges, J., Rischel, E. F. (2022).** "Towards Foundations of Categorical Cybernetics." *Electronic Proceedings in Theoretical Computer Science* 372, 235-248, DOI 10.4204/EPTCS.372.17. — lenses et composition de systèmes dynamiques, prolonge la composition Ghani-Hedges-Winschel-Zahn 2018 aux boucles de rétroaction comme celle de GT-18.
- **Bolt, J., Hedges, J., Zahn, P. (2019).** "Bayesian open games." *Compositionality* 5, article 9, 2023, arXiv:1910.03656, DOI 10.32408/compositionality-5-9. — la composition des *correspondances* d'équilibre, que la Conclusion identifie comme source de la sur-approximation en Surface A.

## À retenir

- **La sélection d'équilibre n'est PAS compositionnelle.** Composer des sélections d'argmax locaux peut produire un profil non-Nash : sur la surface A, le profil `(0, 0)` prédit par la composition naïve cesse d'être un équilibre dès qu'un plateau multivoque rencontre un couplage (écart de déviation `+0.300`).
- **La propriété de Hedges promet la composition des *correspondances*, pas des *sélections*.** Elle sur-approxime l'ensemble des équilibres ; prédire le point exact exige l'univocité — ou une théorie de la sélection.
- **La frontière de contraction est la borne de Banach, exactement** : mesurée à `delt = 2` sans zone grise. Au-delà, le clip borne la divergence en oscillation sur le rail, il ne la répare pas. La boucle GT-18 est le cas dégénéré `L = 1` (convergence-au-rail par le mur, temps borné), distincte de la convergence géométrique de Banach.

## Vérifiez votre compréhension

1. Pourquoi la composition sur-approxime-t-elle l'ensemble des équilibres sans sélectionner l'un d'eux ? Quel est le lien avec le plateau `c = 0.5` ?
2. À `delt = 2` exactement, que prédit la théorie pour la suite des écarts `a_n − b` ? Pourquoi l'amplitude du cycle ne décroît-elle jamais ?
3. En quoi la convergence de la boucle GT-18 est-elle une « convergence-au-rail » et non une convergence géométrique de Banach ? Comment les deux signatures sont-elles distinguées ?

<details>
<summary>Éléments de réponse</summary>

1. La composition des correspondances transporte tous les profils admissibles : elle rend l'ensemble `[(0, 0), (1, 1)]` (cellule 6), sans aucun critère pour préférer l'un des deux. Le plateau `c = 0.5` est exactement ce qui produit cette multivocité : les valeurs locales de J1 sont `-0.250, -0.250, -2.250`, donc `x = 0` et `x = 1` sont à égalité stricte et `BR = {0, 1}` ; la correspondance composée contient les deux profils, elle n'en sélectionne aucun. La sélection est ajoutée par la règle « plus petit argmax », et c'est elle que le jeu composé réfute : le profil `(0, 0)` a un écart de déviation `+0.300` (« NON-NASH du compose »), tandis que `(1, 1)` a `+0.000` (« equilibre du compose »). Sans couplage (`r = 0`), les écarts valent `+0.000, +0.000` : le plateau seul est inoffensif, c'est le retour `r * y` qui rend le choix de `x` payant et fait de l'ensemble une sur-approximation (cellules 8 et 9).

2. À `delt = 2`, `L = |1 - delt| = 1` : l'application n'est ni contractante ni dilatante. Tant que le clip ne mord pas, `a_{n+1} - b = -(a_n - b)` (indice de l'exercice 2), donc la suite des écarts alterne de signe à amplitude constante. L'amplitude ne décroît jamais parce que `|L| = 1` exactement : chaque pas est une réflexion exacte autour de `b`, il n'y a ni amortissement (`L < 1`) ni amplification (`L > 1`). La mesure le confirme à `delt = 2.0` : `L = 1.00`, `a_120 = 0.9000`, `|a_120 - b| = 0.6000`, classé « 2-cycle interieur » (cellule 11), la valeur `0.6` étant l'écart initial `0.9 - 0.3`.

3. La boucle additive de GT-18 (`a' = clip(a + delta)`) a `L = 1` partout et n'est jamais contractante : elle atteint le rail en un temps borné par le mur (1 pas pour `delta = +0.5`, 5 pas pour `delta = -0.3`) et y reste, l'arrêt étant causé par le clip et non par un amortissement interne (cellule 14). Une convergence de Banach a au contraire un taux `|1 - delt| < 1` et un écart qui décroît géométriquement : pour `delt = 0.8` (`L = 0.2`), les écarts mesurés sont `0.6000, 0.1200, 0.0240, 0.0048, 0.0010, 0.0002, 0.0000, 0.0000` avec un ratio moyen de `0.200`, égal à la théorie. Les deux signatures se distinguent donc sur trois traits mesurés dans la cellule 14 : le point d'arrêt (rail `|a| = 1` contre point intérieur `b`), la forme de la suite des écarts (amplitude constante contre décroissance géométrique de taux `|1 - delt|`), et la cause de l'arrêt (le mur contre l'amortissement interne).

</details>

## Pour aller plus loin

- **Sélection d'équilibre dans les jeux composés** à contextes multiples : le voisinage immédiat de ce notebook.
- **Taxonomie des boucles bornées non contractantes** : classifier tous les régimes `L > 1` bornés (le clip borne-t-il toujours en oscillation ?).
- **Kakutani (1941)** — le théorème des correspondances multivoques, base de la multivocité exhibée en surface A.

> Carnet issu de l'EPIC #26534.